In [1]:
import torch
import pandas as pd
import itertools
from sklearn.preprocessing import RobustScaler

In [2]:
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")
train_price = pd.read_csv("train_price.csv", index_col="Date", parse_dates=True)['BTC-USD']
test_price = pd.read_csv("test_price.csv", index_col="Date", parse_dates=True)['BTC-USD']
# Remove the first two rows and reset the index
X_train = X_train.iloc[2:].reset_index(drop=True)
# Rename the "Price" column to "Date"
X_train.rename(columns={"Price": "Date"}, inplace=True)
# Convert the "Date" column to datetime format
X_train["Date"] = pd.to_datetime(X_train["Date"])
# Set "Date" as the index
X_train.set_index("Date", inplace=True)
# Remove the first two rows and reset the index
X_test = X_test.iloc[2:].reset_index(drop=True)
# Rename the "Price" column to "Date"
X_test.rename(columns={"Price": "Date"}, inplace=True)
# Convert the "Date" column to datetime format
X_test["Date"] = pd.to_datetime(X_test["Date"])
# Set "Date" as the index
X_test.set_index("Date", inplace=True)
# Robust scaling
scaler = RobustScaler()
X_train_scaled_array = scaler.fit_transform(X_train)
X_train_scaled = pd.DataFrame(X_train_scaled_array, index=X_train.index, columns=X_train.columns)

X_test_scaled_array = scaler.transform(X_test)
X_test_scaled = pd.DataFrame(X_test_scaled_array, index=X_test.index, columns=X_test.columns)

# Define features
from itertools import combinations
features = [
    list(comb)
    for r in range(1, len(X_train.columns) + 1)
    for comb in combinations(X_train.columns, r)
]

In [3]:
X_train.head()

,Returns,Range,Log_Range,pct_Range,fg,pct_fg,RSI,pct_RSI
Date,,,,,,,,
2018-02-16,0.006618,0.050818,0.049569,-0.431059,67.0,-0.056338,63.799058,0.007788
2018-02-17,0.082383,0.097553,0.093083,0.919639,74.0,0.104478,69.537696,0.089949
2018-02-18,-0.051792,0.099148,0.094535,0.016353,63.0,-0.148649,62.705291,-0.098255
2018-02-19,0.061874,0.072347,0.069850,-0.270310,67.0,0.063492,66.909576,0.067048
2018-02-20,0.015768,0.064700,0.062693,-0.105697,74.0,0.104478,67.940541,0.015408


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import itertools
import time
from tqdm import tqdm

class FeatureExtractor(nn.Module):
    """
    Neural network for feature extraction in PyTorch.
    """
    def __init__(self, input_dim, hidden_dims=[64, 32], embedding_dim=16, dropout_rate=0.2):
        super(FeatureExtractor, self).__init__()
        
        # Build layers
        layers = []
        prev_dim = input_dim
        
        # Add hidden layers
        for dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, dim))
            layers.append(nn.ReLU())
            layers.append(nn.BatchNorm1d(dim))
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = dim
        
        # Add embedding layer
        layers.append(nn.Linear(prev_dim, embedding_dim))
        layers.append(nn.Tanh())
        
        # Combine layers
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)

class HMM:
    """
    Hidden Markov Model implementation using PyTorch.
    """
    def __init__(self, n_states, n_dims, init_type='random'):
        self.n_states = n_states
        self.n_dims = n_dims
        
        # Initialize parameters
        if init_type == 'random':
            # Initial state probabilities (pi)
            self.log_pi = nn.Parameter(torch.randn(n_states).float())
            
            # Transition matrix (A)
            self.log_A = nn.Parameter(torch.randn(n_states, n_states).float())
            
            # Emission parameters for Gaussian emissions
            self.means = nn.Parameter(torch.randn(n_states, n_dims).float())
            # Use log scale for numerical stability
            self.log_vars = nn.Parameter(torch.randn(n_states, n_dims).float())
        else:
            # More stable initialization
            # Initial state probabilities (pi) - uniform
            self.log_pi = torch.log(torch.ones(n_states) / n_states)
            
            # Transition matrix (A) - slightly biased to stay in same state
            A = torch.ones(n_states, n_states) * 0.1
            for i in range(n_states):
                A[i, i] = 0.8  # High self-transition probability
            A = A / A.sum(dim=1, keepdim=True)  # Normalize rows
            self.log_A = torch.log(A)
            
            # Emission parameters - spread out means
            self.means = torch.zeros(n_states, n_dims)
            for i in range(n_states):
                self.means[i] = torch.randn(n_dims) * 2.0 + i  # Separated means
            
            # Start with unit variance
            self.log_vars = torch.zeros(n_states, n_dims)
    
    def normalize_parameters(self):
        """Normalize parameters to ensure valid probabilities."""
        # Normalize initial state probabilities
        log_pi_norm = self.log_pi - torch.logsumexp(self.log_pi, dim=0)
        
        # Normalize transition probabilities
        log_A_norm = self.log_A - torch.logsumexp(self.log_A, dim=1, keepdim=True)
        
        return log_pi_norm, log_A_norm
    
    def emission_log_probs(self, x):
        """
        Calculate emission log probabilities for all states and observations.
        
        Parameters:
        -----------
        x : torch.Tensor
            Observations, shape (batch_size, n_dims)
            
        Returns:
        --------
        torch.Tensor
            Log emission probabilities, shape (batch_size, n_states)
        """
        batch_size = x.shape[0]
        log_emissions = torch.zeros(batch_size, self.n_states)
        
        for j in range(self.n_states):
            # Calculate Gaussian log probability
            vars_j = torch.exp(self.log_vars[j])
            diff = x - self.means[j]
            log_p = -0.5 * (torch.log(2 * np.pi * vars_j) + (diff ** 2) / vars_j)
            log_emissions[:, j] = log_p.sum(dim=1)
        
        return log_emissions
    
    def viterbi_algorithm(self, log_emissions):
        """
        Viterbi algorithm to find the most likely state sequence.
        
        Parameters:
        -----------
        log_emissions : torch.Tensor
            Log emission probabilities, shape (batch_size, n_states)
            
        Returns:
        --------
        torch.Tensor
            Most likely state sequence, shape (batch_size)
        """
        batch_size = log_emissions.shape[0]
        
        # Normalize parameters
        log_pi_norm, log_A_norm = self.normalize_parameters()
        
        # Initialize Viterbi variables
        delta = torch.zeros(batch_size, self.n_states)
        psi = torch.zeros(batch_size, self.n_states, dtype=torch.long)
        
        # Initialization
        delta[0] = log_pi_norm + log_emissions[0]
        
        # Recursion
        for t in range(1, batch_size):
            for j in range(self.n_states):
                probs = delta[t-1] + log_A_norm[:, j]
                psi[t, j] = torch.argmax(probs)
                delta[t, j] = probs[psi[t, j]] + log_emissions[t, j]
        
        # Backtracking
        states = torch.zeros(batch_size, dtype=torch.long)
        states[-1] = torch.argmax(delta[-1])
        
        for t in range(batch_size-2, -1, -1):
            states[t] = psi[t+1, states[t+1]]
        
        return states
    
    def forward_algorithm(self, log_emissions):
        """
        Forward algorithm to calculate log-likelihood.
        
        Parameters:
        -----------
        log_emissions : torch.Tensor
            Log emission probabilities, shape (batch_size, n_states)
            
        Returns:
        --------
        torch.Tensor
            Log-likelihood
        """
        batch_size = log_emissions.shape[0]
        
        # Normalize parameters
        log_pi_norm, log_A_norm = self.normalize_parameters()
        
        # Initialize forward variables
        log_alpha = torch.zeros(batch_size, self.n_states)
        
        # Initialization
        log_alpha[0] = log_pi_norm + log_emissions[0]
        
        # Recursion
        for t in range(1, batch_size):
            for j in range(self.n_states):
                log_alpha[t, j] = torch.logsumexp(log_alpha[t-1] + log_A_norm[:, j], dim=0) + log_emissions[t, j]
        
        # Calculate log-likelihood
        log_likelihood = torch.logsumexp(log_alpha[-1], dim=0)
        
        return log_likelihood
    
    def fit(self, x, max_iter=100, tol=1e-6):
        """
        Fit the HMM using Baum-Welch algorithm.
        
        Parameters:
        -----------
        x : torch.Tensor
            Observations, shape (batch_size, n_dims)
        max_iter : int
            Maximum number of iterations
        tol : float
            Convergence tolerance
            
        Returns:
        --------
        float
            Log-likelihood
        """
        prev_ll = -float('inf')
        
        for iteration in range(max_iter):
            # E-step: Calculate log emission probabilities
            log_emissions = self.emission_log_probs(x)
            
            # Calculate log-likelihood
            log_likelihood = self.forward_algorithm(log_emissions)
            
            # Check convergence
            if log_likelihood - prev_ll < tol:
                break
                
            prev_ll = log_likelihood
            
            # M-step: Need to implement a proper Baum-Welch algorithm here
            # This is a simplified version that just updates means and variances
            
            # Get state assignments (hard assignment for simplicity)
            states = self.viterbi_algorithm(log_emissions)
            
            # Update emission parameters
            for j in range(self.n_states):
                mask = (states == j)
                if mask.sum() > 0:
                    # Update means
                    self.means[j] = x[mask].mean(dim=0)
                    # Update variances (with minimum variance for stability)
                    vars_j = torch.clamp(((x[mask] - self.means[j]) ** 2).mean(dim=0), min=1e-6)
                    self.log_vars[j] = torch.log(vars_j)
            
            # Update transition parameters (simplified)
            state_transitions = torch.zeros((self.n_states, self.n_states))
            for t in range(1, len(states)):
                state_transitions[states[t-1], states[t]] += 1
                
            # Add small value to prevent zeros and normalize
            state_transitions = state_transitions + 1e-6
            state_transitions = state_transitions / state_transitions.sum(dim=1, keepdim=True)
            self.log_A.data = torch.log(state_transitions)
        
        return log_likelihood.item()
    
    def predict(self, x):
        """
        Predict states using Viterbi algorithm.
        
        Parameters:
        -----------
        x : torch.Tensor
            Observations, shape (batch_size, n_dims)
            
        Returns:
        --------
        numpy.ndarray
            Predicted states
        """
        log_emissions = self.emission_log_probs(x)
        states = self.viterbi_algorithm(log_emissions)
        return states.numpy()

def train_neural_network(X, hidden_dims=[64, 32], embedding_dim=16, dropout_rate=0.2, 
                        learning_rate=0.001, epochs=100, batch_size=32):
    """
    Train the neural network feature extractor using PyTorch.
    
    Parameters:
    -----------
    X : numpy.ndarray
        Input features
    hidden_dims, embedding_dim, dropout_rate : model parameters
    learning_rate : float
        Learning rate for the optimizer
    epochs : int
        Number of training epochs
    batch_size : int
        Batch size for training
        
    Returns:
    --------
    tuple
        (trained model, embeddings)
    """
    # Convert to PyTorch tensors
    X_tensor = torch.tensor(X, dtype=torch.float32)
    
    # Create dataset and dataloader
    dataset = TensorDataset(X_tensor, X_tensor)  # Self-supervised
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # Create model
    model = FeatureExtractor(
        input_dim=X.shape[1], 
        hidden_dims=hidden_dims, 
        embedding_dim=embedding_dim, 
        dropout_rate=dropout_rate
    )
    
    # Define loss and optimizer
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # Train the model
    for epoch in range(epochs):
        total_loss = 0
        for inputs, targets in dataloader:
            # Forward pass
            outputs = model(inputs)
            
            # Loss calculation - reconstruction loss (MSE between output and input)
            loss = criterion(outputs, inputs)
            
            # Backward pass and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        # Early stopping could be implemented here
        if epoch % 10 == 0:
            avg_loss = total_loss / len(dataloader)
            print(f"Epoch {epoch}, Loss: {avg_loss:.6f}")
            
            # Early stopping check could go here
    
    # Generate embeddings
    model.eval()
    with torch.no_grad():
        embeddings = model(X_tensor).numpy()
    
    return model, embeddings

def train_pytorch_dnn_hmm(X_subset, is_scaled, n_states=3, hidden_dims=[64, 32], embedding_dim=16, 
                        dropout_rate=0.2, learning_rate=0.001, epochs=100, batch_size=32):
    """
    Train a DNN-HMM model using PyTorch.
    
    Parameters:
    -----------
    X_subset : numpy.ndarray
        Feature data subset
    is_scaled : bool
        Whether the data is already scaled
    n_states, hidden_dims, embedding_dim, etc. : model parameters
        
    Returns:
    --------
    tuple
        (trained_model, log_likelihood, regimes)
    """
    print(f"  Training PyTorch DNN-HMM with X_subset shape: {X_subset.shape}")
    
    # Scale the data if needed
    if not is_scaled:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_subset)
    else:
        scaler = None
        X_scaled = X_subset
    
    # Train neural network and get embeddings
    nn_model, embeddings = train_neural_network(
        X_scaled, 
        hidden_dims=hidden_dims,
        embedding_dim=embedding_dim,
        dropout_rate=dropout_rate,
        learning_rate=learning_rate,
        epochs=epochs,
        batch_size=batch_size
    )
    
    # Create and train HMM
    hmm_model = HMM(n_states=n_states, n_dims=embedding_dim, init_type='stable')
    
    # Convert embeddings to PyTorch tensor
    embeddings_tensor = torch.tensor(embeddings, dtype=torch.float32)
    
    # Train the HMM
    log_likelihood = hmm_model.fit(embeddings_tensor)
    
    # Get state predictions
    regimes = hmm_model.predict(embeddings_tensor)
    
    # Create a composite model containing all necessary components
    model = {
        'hmm_model': hmm_model,
        'nn_model': nn_model,
        'scaler': scaler,
        'input_dim': X_subset.shape[1],
        'embedding_dim': embedding_dim
    }
    
    return model, log_likelihood, regimes

def predict_pytorch_dnn_hmm(model, X_subset):
    """
    Make predictions with a trained PyTorch DNN-HMM model.
    
    Parameters:
    -----------
    model : dict
        Trained model components
    X_subset : numpy.ndarray
        Feature data subset
        
    Returns:
    --------
    numpy.ndarray
        Predicted regimes
    """
    # Extract model components
    hmm_model = model['hmm_model']
    nn_model = model['nn_model']
    scaler = model['scaler']
    
    # Scale the data if needed
    if scaler is not None:
        X_scaled = scaler.transform(X_subset)
    else:
        X_scaled = X_subset
    
    # Convert to PyTorch tensor
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    
    # Generate embeddings
    nn_model.eval()
    with torch.no_grad():
        embeddings = nn_model(X_tensor)
    
    # Make predictions
    try:
        regimes = hmm_model.predict(embeddings)
        return regimes
    except Exception as e:
        print(f"  Error in prediction: {str(e)}")
        # Return default predictions in case of error
        return np.zeros(X_subset.shape[0], dtype=int)

def calculate_silhouette(X_subset, regimes):
    """
    Calculate silhouette score or alternative metric for single feature case.
    
    Parameters:
    -----------
    X_subset : numpy.ndarray
        Feature data subset
    regimes : numpy.ndarray
        Predicted regimes from the model
        
    Returns:
    --------
    float or None
        Silhouette score or alternative metric, None if calculation fails
    """
    # Quick checks before expensive computation
    # Check if we have enough unique regimes
    unique_regimes = np.unique(regimes)
    if len(unique_regimes) <= 1:
        return 0.0
    
    # Special handling for 1D data
    n_features = X_subset.shape[1] if X_subset.ndim > 1 else 1
    
    # For multi-feature data, try standard silhouette score
    if n_features > 1:
        try:
            return silhouette_score(X_subset, regimes)
        except Exception as e:
            print(f"Silhouette score calculation failed: {str(e)}")
            return None
    else:
        # For 1D data, use simplified variance ratio calculation
        try:
            # Reshape 1D data if needed
            X_flat = X_subset.flatten() if X_subset.ndim > 1 else X_subset
            
            # Get overall mean
            overall_mean = np.mean(X_flat)
            overall_var = np.var(X_flat)
            
            if overall_var < 1e-10:  # Effectively zero variance
                return 0.0
            
            # Calculate between and within variance
            between_var = 0
            within_var = 0
            
            # Calculate weighted means and variances for each regime
            for regime in unique_regimes:
                mask = (regimes == regime)
                count = np.sum(mask)
                if count > 0:
                    regime_data = X_flat[mask]
                    regime_mean = np.mean(regime_data)
                    regime_var = np.var(regime_data)
                    
                    # Update between variance
                    between_var += count * (regime_mean - overall_mean) ** 2
                    
                    # Update within variance
                    within_var += count * regime_var
            
            # Normalize by sample size
            between_var /= len(X_flat)
            within_var /= len(X_flat)
            
            # Handle edge cases
            if within_var < 1e-10:  # Effectively zero within-variance
                return 1.0 if between_var > 0 else 0.0
            
            # Calculate ratio and transform to [-1, 1] range
            ratio = between_var / (between_var + within_var)
            return 2 * ratio - 1
            
        except Exception as e:
            print(f"Alternative silhouette calculation failed: {str(e)}")
            return None

def train_pytorch_dnn_hmm_model(X_train, X_train_scaled, features, model_name="PyTorchDNNHMM", n_states=3, 
                              hidden_dims=[64, 32], embedding_dim=16, dropout_rate=0.2,
                              learning_rate=0.001, epochs=100, batch_size=32):
    """
    Train PyTorch DNN-HMM models for all feature combinations.
    
    Parameters:
    -----------
    X_train : pandas.DataFrame
        Raw training data
    X_train_scaled : pandas.DataFrame
        Scaled training data
    features : list of lists
        Feature combinations to use
    model_name, n_states, hidden_dims, etc. : model parameters
        
    Returns:
    --------
    dict
        Dictionary of model information (models_info)
    """
    models_info = {}
    model_id_counter = 0
    
    # Process for both scaling options
    for is_scaled in [False, True]:
        # Select appropriate data
        X = X_train_scaled if is_scaled else X_train
        scaling_name = "robust" if is_scaled else "none"
        
        print(f"\nTraining {model_name} models with {scaling_name} scaling")
        
        # Process each feature combination
        for feature_list in tqdm(features, desc=f"Training models ({scaling_name} scaling)"):
            feature_name = "+".join(feature_list)
            
            try:
                # Extract data for this feature combination
                X_subset = X[feature_list].values
                
                # Train the model
                start_time = time.time()
                model, log_likelihood, regimes = train_pytorch_dnn_hmm(
                    X_subset, 
                    is_scaled,
                    n_states=n_states,
                    hidden_dims=hidden_dims,
                    embedding_dim=embedding_dim,
                    dropout_rate=dropout_rate,
                    learning_rate=learning_rate,
                    epochs=epochs,
                    batch_size=batch_size
                )
                processing_time = time.time() - start_time
                
                # Calculate silhouette score
                silhouette = calculate_silhouette(X_subset, regimes)
                
                # Generate unique model ID
                model_id = f"{model_name}_{scaling_name}_{feature_name}_{model_id_counter}"
                model_id_counter += 1
                
                # Store model information
                models_info[model_id] = {
                    'model': model,
                    'model_type': model_name,
                    'scaling': scaling_name,
                    'features': feature_list,
                    'silhouette_score': silhouette,
                    'log_likelihood': log_likelihood,
                    'regimes': regimes,
                    'processing_time': processing_time
                }
                
                # Debug output
                print(f"Trained model with features {feature_list}, silhouette: {silhouette}, log-likelihood: {log_likelihood}")
                
            except Exception as e:
                print(f"Error training model with features {feature_list}, scaling {scaling_name}: {str(e)}")
    
    return models_info

def predict_with_model(models_info, model_id, X_new):
    """
    Make predictions with a specific model.
    
    Parameters:
    -----------
    models_info : dict
        Dictionary containing all trained models
    model_id : str
        ID of the model to use for prediction
    X_new : pandas.DataFrame
        New data for prediction
        
    Returns:
    --------
    numpy.ndarray
        Predicted regimes
    """
    if model_id not in models_info:
        raise ValueError(f"Model ID {model_id} not found")
    
    # Get model information
    model_info = models_info[model_id]
    model = model_info['model']
    features = model_info['features']
    
    # Extract features for prediction
    X_subset = X_new[features].values
    
    # Make prediction
    return predict_pytorch_dnn_hmm(model, X_subset)

def get_best_models(models_info, metric='silhouette_score', top_n=5, min_features=1, max_features=None):
    """
    Get the best performing models based on a specified metric.
    
    Parameters:
    -----------
    models_info : dict
        Dictionary containing all trained models
    metric : str
        Metric to use for ranking ('silhouette_score' or 'log_likelihood')
    top_n : int
        Number of top models to return
    min_features, max_features : feature count constraints
        
    Returns:
    --------
    list
        List of (model_id, score) tuples sorted by score
    """
    # Filter models by feature count
    filtered_models = []
    
    for model_id, info in models_info.items():
        feature_count = len(info['features'])
        if feature_count >= min_features and (max_features is None or feature_count <= max_features):
            # Check if the metric exists and is not None
            if metric in info and info[metric] is not None:
                filtered_models.append((model_id, info[metric]))
    
    # Sort by metric (higher is better for silhouette, log_likelihood)
    sorted_models = sorted(filtered_models, key=lambda x: x[1], reverse=True)
    
    # Return top N models
    return sorted_models[:top_n]

def export_results(models_info, filename):
    """
    Export key results in a format suitable for analysis.
    Creates a CSV file with model results.
    
    Parameters:
    -----------
    models_info : dict
        Dictionary containing all trained models
    filename : str
        File path to save the results
        
    Returns:
    --------
    bool
        Success status
    """
    try:
        import pandas as pd
        
        # Create results dataframe
        results = []
        
        for model_id, info in models_info.items():
            # Extract basic information
            result = {
                'model_id': model_id,
                'model_type': info['model_type'],
                'scaling': info['scaling'],
                'features': '+'.join(info['features']),
                'feature_count': len(info['features']),
                'silhouette_score': info['silhouette_score'],
                'log_likelihood': info['log_likelihood'],
                'processing_time': info.get('processing_time', None)
            }
            
            results.append(result)
        
        # Convert to dataframe and save
        df = pd.DataFrame(results)
        df.to_csv(filename, index=False)
        
        print(f"Exported {len(results)} model results to {filename}")
        return True
    except Exception as e:
        print(f"Error exporting results: {str(e)}")
        return False

def create_dummy_hmm_object(models_info, model_name="PyTorchDNNHMM", n_states=3):
    """
    Create a dummy object that mimics a BaseHMMModel instance.
    This allows us to use the existing HMMLeaderboard and HMMVisualizer classes.
    
    Parameters:
    -----------
    models_info : dict
        Dictionary of model information from train_pytorch_dnn_hmm_model()
    model_name : str
        Name for this model type
    n_states : int
        Number of hidden states
        
    Returns:
    --------
    object
        An object with the necessary attributes to work with leaderboard/visualizer
    """
    class DummyHMM:
        def __init__(self, models_info, model_name, n_states):
            self.models_info = models_info
            self.model_name = model_name
            self.n_states = n_states
            
        def predict_with_model(self, model_id, X_new):
            """Forward the prediction to our function"""
            return predict_with_model(self.models_info, model_id, X_new)
    
    return DummyHMM(models_info, model_name, n_states)

# Example usage:
"""
# Create feature combinations
def create_feature_combinations(columns):
    features = []
    for r in range(1, len(columns) + 1):
        combinations = [list(combo) for combo in itertools.combinations(columns, r)]
        features.extend(combinations)
    return features

# Select features and create combinations
selected_columns = ['Range', 'fg', 'RSI']
X_train_selected = X_train[selected_columns]
features = create_feature_combinations(selected_columns)

# Train PyTorch DNN-HMM models
pytorch_dnn_hmm_models = train_pytorch_dnn_hmm_model(
    X_train_selected, 
    X_train_scaled, 
    features,
    model_name="PyTorchDNNHMM",
    n_states=3,
    hidden_dims=[32, 16],
    embedding_dim=8,
    epochs=50
)

# Create dummy HMM object for leaderboard and visualizer
dnn_hmm = create_dummy_hmm_object(pytorch_dnn_hmm_models)

# Get best models
best_models = get_best_models(pytorch_dnn_hmm_models, top_n=3)
print("Best models:", best_models)

# Make predictions with the best model
best_model_id = best_models[0][0]
predictions = predict_with_model(pytorch_dnn_hmm_models, best_model_id, X_test)

# Export results
export_results(pytorch_dnn_hmm_models, "pytorch_dnn_hmm_results.csv")
"""

In [ ]:
# Import the PyTorch DNN-HMM functions
from pytorch_dnn_hmm_functions import (
    train_pytorch_dnn_hmm_model,
    predict_with_model,
    get_best_models,
    export_results,
    create_dummy_hmm_object
)

